# Chapter 05: Text Clustering with Embeddings

Clustering groups similar documents together **without labels**. We embed text into vectors, then run k-means to find natural groups.

```
[Doc1] [Doc2] [Doc3] ...  →  embed  →  [vec1] [vec2] [vec3]  →  cluster  →  Group A / Group B
```

## Step 1 — Install dependencies

In [ ]:
# !pip install sentence-transformers scikit-learn matplotlib
print('Dependencies ready')

## Step 2 — Load sample documents

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# Load 4 topic categories so we can verify clusters make sense
cats = ["sci.space", "rec.sport.hockey", "talk.politics.guns", "comp.graphics"]
data = fetch_20newsgroups(subset="train", categories=cats, remove=("headers","footers","quotes"))
docs = data.data[:200]   # 200 docs to keep it fast
labels_true = data.target[:200]
print(f"Loaded {len(docs)} documents across {len(cats)} topics")
print("First doc snippet:", docs[0][:120])

## Step 3 — Embed documents

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")   # fast 384-dim model
embeddings = model.encode(docs, batch_size=32, show_progress_bar=True)
print("Embeddings shape:", embeddings.shape)   # (200, 384)

## Step 4 — Cluster with k-means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

k = 4   # we know there are 4 topics
km = KMeans(n_clusters=k, random_state=42, n_init=10)
cluster_labels = km.fit_predict(embeddings)

ari = adjusted_rand_score(labels_true, cluster_labels)
print(f"Adjusted Rand Index: {ari:.3f}  (1.0 = perfect match)")

## Step 5 — Visualise with UMAP / PCA

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2, random_state=42)
pts = pca.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter_kw = dict(s=15, alpha=0.7)

axes[0].scatter(pts[:,0], pts[:,1], c=labels_true, cmap="tab10", **scatter_kw)
axes[0].set_title("True topic labels")

axes[1].scatter(pts[:,0], pts[:,1], c=cluster_labels, cmap="tab10", **scatter_kw)
axes[1].set_title(f"K-Means clusters  (ARI={ari:.2f})")

plt.tight_layout()
plt.savefig("cluster_plot.png", dpi=120)
plt.show()
print("Plot saved to cluster_plot.png")

## Step 6 — Inspect cluster keywords

In [ ]:
from collections import Counter
import re

def top_words(indices, n=8):
    words = []
    for i in indices:
        words += re.findall(r"\b[a-z]{4,}\b", docs[i].lower())
    stopwords = {"this","that","with","from","have","they","were","been","will","their","which"}
    filtered = [w for w in words if w not in stopwords]
    return [w for w,_ in Counter(filtered).most_common(n)]

for cid in range(k):
    idxs = [i for i,l in enumerate(cluster_labels) if l==cid]
    print(f"Cluster {cid} ({len(idxs)} docs): {top_words(idxs)}")

## Summary

| Step | What happened |
|------|---------------|
| Embed | `all-MiniLM-L6-v2` mapped each doc to a 384-dim vector |
| Cluster | K-Means grouped vectors into 4 clusters |
| Evaluate | ARI measures how well clusters match true topics |

Try changing `k` to 8 or 10 and observe how the ARI changes.